In [6]:
import sys

sys.path.append("../")

from sls_client import get_sls_data_by_query
from datetime import datetime,timedelta

today=datetime.now()
two_month_ago=today-timedelta(days=60)

print(two_month_ago, today)

query="""upstream_status:200 and proxy_upstream_name:product-backend-mall-svc-80 and not url:'/order/PO25' 
and not url:expand_activity and not url:/timing-delivery/sku/ and not url:/register/claim/
| select split_part(regexp_replace(url, '/\d[A-Z0-9]*','/{param}'),'?',1) as api,
count(distinct xm_rqid) request_cnt,count(distinct xm_phone) users 
from log group by 1 having api not like '/order/PO25%' order by 2 desc limit 10000"""

api_request_df=get_sls_data_by_query(from_time=two_month_ago, to_time=today, query=query, logstore="nginx-ingress")

2025-01-06 18:24:00.361007 2025-03-07 18:24:00.361007
即将获取数据: =====> 2025-01-06 18:24:00.361007 2025-03-07 18:24:00.361007 nginx-ingress: upstream_status:200 and proxy_upstream_name:product-backend-mall-svc-80 and not url:'/order/PO25' 
a
>=====数条数:207


In [7]:
api_request_df.head(2)

,api,request_cnt,users,__source__,__time__
0,/price/query/take-actual-price,3658956,69373,,1736159040
1,/mall/tag/query/all,3067961,70346,,1736159040


In [25]:
import json
import pandas as pd

# 从文件中加载 OpenAPI 规范
with open("./openapi.json") as f:
    openapi_spec = json.load(f)

# 提取 paths 字段，并将其转换为 DataFrame
mall_apis = openapi_spec.get('paths', {})
mall_apis_df = pd.DataFrame(list(mall_apis.items()), columns=['api', 'detail'])

# 清理 API 路径，将常见的参数模式替换为 {params}
import re

def replace_params(api_path):
    # 定义需要替换的参数模式
    patterns = [
        r'\{orderNo\}',
        r'\{pageSize\}',
        r'\{pageIndex\}',
        r'\{afterSaleOrderNo\}',
        r'\{mId\}',
        # 添加更多可能的参数模式
    ]
    # 逐个应用替换规则
    for pattern in patterns:
        api_path = re.sub(pattern, '{param}', api_path)
    return api_path.split(',')[0]

mall_apis_df['api'] = mall_apis_df['api'].apply(replace_params)

# 将 mall_apis_df 与 api_request_df 进行左连接，api_request_df 作为右表
merged_df = pd.merge(mall_apis_df, api_request_df, on='api', how='left')

# 显示合并后的 DataFrame 的前几行
merged_df.head(2)

,api,detail,request_cnt,users,__source__,__time__
0,/openid,"{'get': {'summary': '微信登录', 'description': '微信...",311403,26736,,1736159040
1,/loginV2,"{'post': {'summary': 'V2登录接口', 'description': ...",4240,226,,1736159040


In [26]:
merged_df["users"] = merged_df["users"].fillna(0)
merged_df["users"] = merged_df["users"].astype(int)
used_apis_df=merged_df[merged_df["users"] > 0]
used_apis_df

,api,detail,request_cnt,users,__source__,__time__
0,/openid,"{'get': {'summary': '微信登录', 'description': '微信...",311403,26736,,1736159040
1,/loginV2,"{'post': {'summary': 'V2登录接口', 'description': ...",4240,226,,1736159040
2,/login-info,"{'get': {'summary': '获取登录信息', 'description': '...",243741,7794,,1736159040
3,/register/wechat-code,"{'get': {'summary': '微信容器打开页面url获取', 'descript...",24264,4,,1736159040
4,/register/pop-wechat-code,"{'get': {'summary': '微信容器打开页面url获取', 'descript...",158,1,,1736159040
...,...,...,...,...,...,...
242,/validityCheck,"{'post': {'summary': '消息的接收、处理、响应', 'descripti...",1,1,,1736159040
243,/JssdkConfig,"{'post': {'summary': '获取用户凭证ticket', 'descript...",383444,57718,,1736159040
244,/popJssdkConfig,"{'post': {'summary': '获取pop用户凭证ticket', 'descr...",4525,408,,1736159040
247,/wechatpay/JsapiPay,"{'post': {'summary': '调起支付-统一下单接口-公众号-小程序', 'd...",4,1,,1736159040


In [29]:
# 筛选出未使用的 API
unsued_df = merged_df[merged_df["users"] == 0]

# 定义一个函数来提取 summary 和 description
def extract_details(detail):
    # 检查 detail 是否为字典，以及是否包含 'get' 或 'post' 键
    if isinstance(detail, dict):
        # 尝试获取 'get' 方法的详情
        get_details = detail.get('get')
        if get_details:
            return get_details.get('summary', ''), get_details.get('description', '')
        # 如果没有 'get'，尝试获取 'post' 方法的详情
        post_details = detail.get('post')
        if post_details:
            return post_details.get('summary', ''), post_details.get('description', '')
    return '', ''  # 如果没有找到 'get' 或 'post'，返回空字符串

# 应用函数以提取 summary 和 description，并创建新的列
unsued_df[['summary', 'description']] = unsued_df['detail'].apply(lambda x: pd.Series(extract_details(x)))

# 根据 'api' 列排序
unsued_df = unsued_df.sort_values(by='api')
unsued_df[['api','detail','summary']].to_csv("./mall60天以内都未有调用的接口.csv", index=False)

/var/folders/b3/9hcz86fx1_z_8m4121xwbs2h0000gn/T/ipykernel_22742/3937582056.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  unsued_df[['summary', 'description']] = unsued_df['detail'].apply(lambda x: pd.Series(extract_details(x)))
/var/folders/b3/9hcz86fx1_z_8m4121xwbs2h0000gn/T/ipykernel_22742/3937582056.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  unsued_df[['summary', 'description']] = unsued_df['detail'].apply(lambda x: pd.Series(extract_details(x)))
